# Move fields containing ★ into Extra2 via AnkiConnect

This notebook connects to Anki through the **AnkiConnect** plugin and does the following for a chosen note type:

- scans every field in each note
- if a field contains `★`, moves that field's full content into `Extra2` as `FieldName--` + content
- clears the original source field

Run the setup cell first. Keep `DRY_RUN = True` for a preview, then set it to `False` to apply changes.

In [ ]:
import json
import urllib.request
from typing import Any, Dict, List

ANKI_CONNECT_URL = "http://127.0.0.1:8765"

def invoke(action: str, **params) -> Any:
    payload = json.dumps({"action": action, "version": 6, "params": params}).encode("utf-8")
    req = urllib.request.Request(ANKI_CONNECT_URL, data=payload, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req) as resp:
        body = json.loads(resp.read().decode("utf-8"))
    if body.get("error"):
        raise RuntimeError(f"AnkiConnect error in {action}: {body['error']}")
    return body.get("result")

# Quick connection check
print("AnkiConnect version:", invoke("version"))

In [ ]:
# Configuration
NOTE_TYPE = "Your Note Type Name"  # e.g. "Migaku Japanese"
DEST_FIELD = "Extra2"
SYMBOL = "★"
SEPARATOR = "\n\n"
DRY_RUN = True

In [ ]:
def transform_note_fields(fields: Dict[str, Dict[str, str]]) -> Dict[str, Any]:
    if DEST_FIELD not in fields:
        return {"skip": "missing_dest", "updates": {}, "moved": []}

    moved_contents: List[str] = []
    updates: Dict[str, str] = {}

    for fname, fobj in fields.items():
        if fname == DEST_FIELD:
            continue
        value = fobj.get("value", "")
        if SYMBOL in value:
            moved_contents.append(f"{fname}--\n{value}")
            updates[fname] = ""

    if not moved_contents:
        return {"skip": "no_symbol", "updates": {}, "moved": []}

    dest_current = fields[DEST_FIELD].get("value", "")
    moved_payload = SEPARATOR.join(moved_contents).strip()

    if dest_current.strip():
        updates[DEST_FIELD] = dest_current.rstrip() + SEPARATOR + moved_payload
    else:
        updates[DEST_FIELD] = moved_payload

    return {"skip": None, "updates": updates, "moved": moved_contents}

# Find notes by note type
query = f'note:"{NOTE_TYPE}"'
nids = invoke("findNotes", query=query)
print(f"Found {len(nids)} notes for note type: {NOTE_TYPE}")

if not nids:
    raise RuntimeError("No notes found. Check NOTE_TYPE spelling.")

# Process in chunks
chunk_size = 200
notes_changed = 0
fields_moved = 0
missing_dest = 0
preview = []

for i in range(0, len(nids), chunk_size):
    chunk = nids[i:i + chunk_size]
    infos = invoke("notesInfo", notes=chunk)

    for info in infos:
        nid = info["noteId"]
        result = transform_note_fields(info["fields"])

        if result["skip"] == "missing_dest":
            missing_dest += 1
            continue
        if result["skip"] == "no_symbol":
            continue

        notes_changed += 1
        fields_moved += len(result["moved"])

        if len(preview) < 5:
            preview.append({
                "noteId": nid,
                "moved_fields_count": len(result["moved"]),
                "new_extra2_prefix": result["updates"][DEST_FIELD][:120],
            })

        if not DRY_RUN:
            invoke("updateNoteFields", note={"id": nid, "fields": result["updates"]})

print({
    "dry_run": DRY_RUN,
    "notes_changed": notes_changed,
    "fields_moved": fields_moved,
    "missing_dest_field_notes": missing_dest,
})

print("Preview (first 5 changed notes):")
for p in preview:
    print(p)

## Apply changes

1. Confirm the dry-run output looks correct.
2. Set `DRY_RUN = False` in the config cell.
3. Re-run the processing cell to write updates into Anki.

Recommended: create an Anki backup before applying changes.